In [2]:
import importlib
import weights_cuda
import spike_engine_cuda
importlib.reload(weights_cuda)
importlib.reload(spike_engine_cuda)
from spike_engine_cuda import SpikeEngineCUDA
from topologies import square_torus
import cupy as cp


In [8]:
# N = 50, lifetime = 100000 # can it finish in a minute?
N = 256
lifetime = 1000
engine = SpikeEngineCUDA(
    square_torus(N),
    (N, N),
    use_k2tree=True,
    verify_k2tree=False,
    verify_progress_every=2000,
    decay_rate=0.66,
    rank=1
)


Constructing weight matrix...
Weights constructed.


In [ ]:
# Estimate bifurcation threshold and set constant weights near it
w_accum, w_instant = engine.estimate_bifurcation_weight(input_period=1)
target, _, _ = engine.set_constant_weights_near_bifurcation(input_period=1, scale=1.2, freeze_learning=True)
print(f"w_accum={w_accum:.6f} w_instant={w_instant:.6f} target={target:.6f}")


In [12]:
input_neuron = (N * N) // 2 + N//2
engine.set_input_neurons([input_neuron])
inputspikes = cp.ones((lifetime, 1))

engine.start_static_record(inputspikes, lifetime, "cuda_test_9.spire")
# rsync -avP user@remote:/path/to/file /local/path


100%|█████████████████████████████████████████████████████████████████████████| 1000/1000 [00:03<00:00, 310.47it/s]

Recording saved: cuda_test_9.spire


In [ ]:
# Optional: validate neighbors for a random neuron
idx = 123
print("neighbors:", engine.weights.get_neighbors(idx))
